# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset:

**"Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"**, using the [mlcroissant](https://croissant.sen.science/) library. We will access data described in Croissant schema, review its record sets and fields, and perform basic exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This includes the package description and mechanisms to interact with record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
pd.set_option('display.max_columns', None)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Let's examine the record sets and fields that are available to us in the dataset via their `@id` values. We'll enumerate record sets, the fields they contain, and their types.

In [ ]:
# List available record sets and their field ids and types
record_sets = list(dataset.record_sets)

if not record_sets:
    print("This dataset does not explicitly declare record sets.\nHowever, mlcroissant exposes record sets defined by data files.\nLet's auto-list any accessible record set ids via dataset.record_sets:")
    # Let's inspect underlying structure
    record_sets = list(dataset.record_sets)
    if not record_sets:
        print("No record sets found via Croissant metadata.")
    else:
        for record_set in record_sets:
            print(f"Record set id: {getattr(record_set, '@id', None)}  —  Name: {getattr(record_set, 'name', None)}")
            if hasattr(record_set, 'fields') and record_set.fields:
                print("  Fields:")
                for fld in record_set.fields:
                    fid = getattr(fld, '@id', None)
                    fname = getattr(fld, 'name', None)
                    ftype = getattr(fld, 'dataType', None)
                    print(f"    - {fid} ({fname}), type: {ftype}")
            print()
else:
    for record_set in record_sets:
        print(f"Record set id: {getattr(record_set, '@id', None)}  —  Name: {getattr(record_set, 'name', None)}")
        if hasattr(record_set, 'fields') and record_set.fields:
            print("  Fields:")
            for fld in record_set.fields:
                fid = getattr(fld, '@id', None)
                fname = getattr(fld, 'name', None)
                ftype = getattr(fld, 'dataType', None)
                print(f"    - {fid} ({fname}), type: {ftype}")

Let's try to enumerate and sample records for all found record sets using their `@id`.

In [ ]:
# Attempt to list records for each detected record set
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"\n➡️ Record Set: {getattr(record_set, '@id', None)}  -  {getattr(record_set, 'name', None)}")
    try:
        firsts = list(dataset.records(record_set=getattr(record_set, '@id', None)))[:2]
        for rec in firsts:
            print(rec)
    except Exception as e:
        print(f"Error reading records from {getattr(record_set, '@id', None)}: {str(e)}")

## 3. Data Extraction
We will now load the data from all available record sets into pandas DataFrames for further analysis.

**Note:** For this dataset, the main record set is typically the largest data file representing the main table of protein attributes.

In [ ]:
# Gather all record_set @ids
all_record_set_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets]

dataframes = {}
for recset_id in all_record_set_ids:
    try:
        recs = list(dataset.records(record_set=recset_id))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[recset_id] = df
            print(f"Loaded {len(df)} records from record set {recset_id}.")
        else:
            print(f"No records found for record set {recset_id}.")
    except Exception as e:
        print(f"Error for record set {recset_id}: {e}")

# List the main table
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Heuristic: first one is likely main
    print(f"Main record set: {main_record_set_id}")
    print("Columns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular records loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's process the main table. Below, we will:
- Select a numeric field for analysis (e.g., `MW [kDa]`, the molecular weight column, if present)
- Filter records where molecular weight > 10 kDa
- Normalize the molecular weight
- Optionally, group by a categorical field, such as a protein description or another attribute

All fields/columns are referenced by their `@id` value, which is typically their Croissant property (column header or id).

In [ ]:
main_rs_id = main_record_set_id if dataframes else None
df = dataframes.get(main_rs_id, pd.DataFrame())

if not df.empty:
    # Try to detect a numeric column:
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64','int64']]
    if not numeric_candidates:
        # Try to coerce some likely candidates:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        # Filter records
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (showing first rows):")
        display(filtered_df.head())
        # Normalize field (z-score)
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
        # Try to group by a categorical field
        group_field = None
        # Example: find a field called 'Description' or similar
        for col in df.columns:
            if 'description' in col.lower() or 'name' in col.lower() or 'protein' in col.lower():
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped (mean) {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No appropriate categorical field for grouping detected.")
    else:
        print("No numeric fields found in the main DataFrame.")
else:
    print("Main record set DataFrame is empty.")

## 5. Visualization
Let's plot the distribution of the chosen numeric field, and a scatterplot relating two numeric properties (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_candidates:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    # Try a scatterplot
    if len(numeric_candidates) > 1:
        plt.figure(figsize=(6,6))
        other_numeric = numeric_candidates[1]
        sns.scatterplot(data=df, x=numeric_field, y=other_numeric)
        plt.xlabel(numeric_field)
        plt.ylabel(other_numeric)
        plt.title(f"{numeric_field} vs {other_numeric}")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² protein mass spectrometry dataset, explored its metadata, discovered and viewed its tabular structure, performed basic filtering and normalization of a numeric field, and visualized its distribution. This demonstrates how to access, inspect, and prepare Croissant-based biomedical data for downstream analysis using the `mlcroissant` library.

Further steps could include deeper analysis, advanced data cleaning or biological interpretation with domain knowledge, and exporting processed data for use in machine learning pipelines.